# Bangladesh Urban Center Mapping — ALL ONLINE

This notebook follows the **interactive Earth Search tutorial style**, but covers the **whole of Bangladesh** and implements the multi-factor urban-center workflow.

### Online sources
- **Sentinel-2:** Element84 Earth Search STAC
- **VIIRS Nighttime Lights:** Google Earth Engine
- **Population (2025):** GHSL population via Google Earth Engine
- **Topography:** Copernicus DEM GLO-30 via Google Earth Engine
- **Roads:** Overture Maps cloud GeoParquet via DuckDB
- **POI / Services:** Overture Maps Places cloud GeoParquet via DuckDB

No manual download of these factor datasets is required. Data are queried from the cloud inside VS Code/Jupyter. Only final outputs are written to your `outputs` folder.

> Earth Engine requires a one-time authentication and a Google Cloud project enabled for Earth Engine.


In [ ]:
# 0. INSTALL PACKAGES (RUN ONCE IN A CLEAN CONDA ENVIRONMENT)
#
# Recommended:
#
# conda create -n urban_center -c conda-forge --override-channels ^
#   python=3.11 geopandas rasterio gdal proj pyproj rioxarray xarray dask ^
#   pystac-client stackstac leafmap shapely scipy matplotlib pandas numpy ^
#   pyogrio duckdb earthengine-api geemap xee ipykernel -y
#
# Then:
#
# conda activate urban_center
# python -m ipykernel install --user --name urban_center --display-name "Python (urban_center)"
#
# In VS Code select kernel: Python (urban_center)


In [ ]:
# 1. CONFIGURATION
from pathlib import Path
import os

PROJECT_DIR = Path(r"E:\Geospatial\Urban Center\Urban-Center")
AOI_PATH = PROJECT_DIR / "bgd_admin_boundaries.shp" / "bgd_admin0.shp"
OUTPUT_DIR = PROJECT_DIR / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Earth Engine project:
# Replace this with YOUR Earth Engine-enabled Google Cloud project ID.
EE_PROJECT = "solid-garden-458417-t4"

# Sentinel-2 time window
S2_DATE_RANGE = "2025-01-01/2025-03-31"
S2_START = "2025-01-01"
S2_END = "2025-04-01"
MAX_CLOUD = 10
N_BEST_PER_TILE = 5

# VIIRS persistent light year
VIIRS_START = "2025-01-01"
VIIRS_END = "2026-01-01"

# Analysis grid
TARGET_EPSG = 6933
RESOLUTION_M = 100
CHUNK_SIZE = 512

# Overture current stable release used in this notebook.
OVERTURE_RELEASE = "2026-06-17.0"

WEIGHTS = {
    "builtup": 0.30,
    "nightlight": 0.20,
    "population": 0.20,
    "road": 0.10,
    "poi": 0.10,
    "topography": 0.10,
}

URBAN_SCORE_THRESHOLD = 0.55
MIN_PATCH_AREA_KM2 = 1.0
MIN_MEAN_POP_DENSITY = 500.0
MIN_MEAN_URBAN_SCORE = 0.60

os.environ["GDAL_HTTP_MAX_RETRY"] = "5"
os.environ["GDAL_HTTP_RETRY_DELAY"] = "2"
os.environ["GDAL_HTTP_RETRY_CODES"] = "ALL"
os.environ["GDAL_HTTP_TCP_KEEPALIVE"] = "YES"

print("AOI:", AOI_PATH)
print("Outputs:", OUTPUT_DIR)
print("Overture release:", OVERTURE_RELEASE)


In [ ]:
# CONFIG / CONSTANTS

TARGET_EPSG = 6933
RESOLUTION_M = 100
CHUNK_SIZE = 512

print("TARGET_EPSG:", TARGET_EPSG)

In [ ]:
# 2. IMPORTS + LOCAL GEO ENVIRONMENT CHECK
from collections import defaultdict, Counter
import warnings
import time

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt

import rasterio
from rasterio.errors import RasterioIOError
from rasterio.features import rasterize, shapes
from rasterio.enums import MergeAlg
import pyproj

import xarray as xr
import rioxarray
import stackstac
from pystac_client import Client

from shapely.geometry import shape, Point
from shapely.ops import unary_union
from shapely import from_wkb

from scipy import ndimage
import leafmap
import duckdb

warnings.filterwarnings("ignore", category=FutureWarning)

print("Rasterio:", rasterio.__version__)
print("GDAL:", rasterio.__gdal_version__)
print("PROJ:", pyproj.proj_version_str)
print("StackSTAC:", stackstac.__version__)

# This catches the PROJ mismatch you previously encountered.
try:
    print("CRS test:", rasterio.crs.CRS.from_epsg(TARGET_EPSG))
except Exception as exc:
    raise RuntimeError(
        "Your GDAL/PROJ environment is broken. "
        "Create the clean 'urban_center' conda environment from Cell 0, "
        "then select that kernel in VS Code.\n\n"
        f"Original error: {exc}"
    )


In [ ]:
# 3. LOAD BANGLADESH AOI
if not AOI_PATH.exists():
    raise FileNotFoundError(
        f"AOI not found:\n{AOI_PATH}\n"
        "Edit PROJECT_DIR / AOI_PATH in Cell 1."
    )

bd = gpd.read_file(AOI_PATH)

if bd.empty:
    raise ValueError("Bangladesh AOI is empty.")
if bd.crs is None:
    raise ValueError("Bangladesh AOI has no CRS.")

bd = bd.to_crs(4326)
aoi = bd[["geometry"]].dissolve().reset_index(drop=True)

if not aoi.geometry.iloc[0].is_valid:
    aoi["geometry"] = aoi.geometry.buffer(0)

bd_geom = aoi.geometry.iloc[0]
west, south, east, north = map(float, aoi.total_bounds)

print("CRS:", aoi.crs)
print("Bounds:", (west, south, east, north))
print("AOI valid:", bd_geom.is_valid)


In [ ]:
# 4. SHOW WHOLE BANGLADESH IN LEAFMAP
m = leafmap.Map(
    center=[23.6850, 90.3563],
    zoom=7,
    height="700px",
)

m.add_gdf(
    aoi,
    layer_name="Bangladesh AOI",
    style={
        "color": "red",
        "weight": 3,
        "fillColor": "red",
        "fillOpacity": 0.04,
    },
)

m


## Part A — Sentinel-2 from Earth Search

This keeps the same overall approach as the Earth Search tutorial, but replaces the single point with the **Bangladesh national bounding box**, followed by exact local intersection with the Bangladesh polygon.


In [ ]:
# 5. SEARCH SENTINEL-2 — EARTH SEARCH
catalog = Client.open("https://earth-search.aws.element84.com/v1")

search = catalog.search(
    collections=["sentinel-2-c1-l2a"],
    bbox=[west, south, east, north],
    datetime=S2_DATE_RANGE,
    query={"eo:cloud_cover": {"lt": MAX_CLOUD}},
)

items_raw = list(search.items())

print("Raw bbox results:", len(items_raw))

items_bd = []

for item in items_raw:
    if item.geometry is None:
        continue
    try:
        if shape(item.geometry).intersects(bd_geom):
            items_bd.append(item)
    except Exception as exc:
        print("Skipped:", item.id, type(exc).__name__)

if not items_bd:
    raise RuntimeError("No Sentinel-2 scenes intersect Bangladesh.")

dates = sorted({
    item.datetime.strftime("%Y-%m-%d")
    for item in items_bd
    if item.datetime is not None
})

print("Scenes intersecting Bangladesh:", len(items_bd))
print("Unique acquisition dates:", len(dates))
print("First dates:", dates[:10])


In [ ]:
# 6. SHOW ALL SEARCHED SENTINEL-2 FOOTPRINTS
scene_gdf = gpd.GeoDataFrame(
    [
        {
            "id": item.id,
            "cloud": item.properties.get("eo:cloud_cover"),
            "geometry": shape(item.geometry),
        }
        for item in items_bd
    ],
    crs="EPSG:4326",
)

m_scenes = leafmap.Map(
    center=[23.6850, 90.3563],
    zoom=7,
    height="700px",
)

m_scenes.add_gdf(
    scene_gdf,
    layer_name="Sentinel-2 footprints",
    style={
        "color": "blue",
        "weight": 1,
        "fillColor": "blue",
        "fillOpacity": 0.04,
    },
)

m_scenes.add_gdf(
    aoi,
    layer_name="Bangladesh",
    style={"color": "red", "weight": 3, "fillOpacity": 0},
)

m_scenes


In [ ]:
# 7. SELECT LOW-CLOUD SCENES + GUARANTEE 100% FOOTPRINT COVERAGE

def get_tile_id(item):
    parts = item.id.split("_")
    if len(parts) > 1 and parts[1].startswith("T"):
        return parts[1]

    grid_code = item.properties.get("grid:code")
    if grid_code:
        return str(grid_code)

    raise ValueError(f"Cannot identify tile for {item.id}")


items_by_tile = defaultdict(list)

for item in items_bd:
    try:
        items_by_tile[get_tile_id(item)].append(item)
    except ValueError as exc:
        print(exc)


selected_items = []

for tile_id, tile_items in items_by_tile.items():
    tile_items = sorted(
        tile_items,
        key=lambda item: (
            item.properties.get("eo:cloud_cover", 100),
            item.datetime.isoformat() if item.datetime else "",
        ),
    )
    selected_items.extend(tile_items[:N_BEST_PER_TILE])


def union_footprints(stac_items):
    geoms = [
        shape(item.geometry)
        for item in stac_items
        if item.geometry is not None
    ]
    return unary_union(geoms)


selected_union = union_footprints(selected_items)
missing_geom = bd_geom.difference(selected_union)

selected_ids = {item.id for item in selected_items}

remaining_items = sorted(
    [item for item in items_bd if item.id not in selected_ids],
    key=lambda item: item.properties.get("eo:cloud_cover", 100),
)

extra_items = []

for item in remaining_items:
    if missing_geom.is_empty:
        break

    scene_geom = shape(item.geometry)
    gain = missing_geom.intersection(scene_geom)

    if not gain.is_empty and gain.area > 0:
        selected_items.append(item)
        extra_items.append(item)
        selected_union = selected_union.union(scene_geom)
        missing_geom = bd_geom.difference(selected_union)


coverage_check = gpd.GeoDataFrame(
    {"kind": ["aoi", "covered"]},
    geometry=[
        bd_geom,
        bd_geom.intersection(selected_union),
    ],
    crs="EPSG:4326",
).to_crs(TARGET_EPSG)

coverage_pct = (
    coverage_check.geometry.iloc[1].area
    / coverage_check.geometry.iloc[0].area
    * 100
)

print("Unique MGRS tiles:", len(items_by_tile))
print("Selected scenes:", len(selected_items))
print("Extra scenes added:", len(extra_items))
print(f"Bangladesh footprint coverage: {coverage_pct:.6f}%")


In [ ]:
# 8. SHOW FINAL SELECTED FOOTPRINTS
selected_scene_gdf = gpd.GeoDataFrame(
    [
        {
            "id": item.id,
            "cloud": item.properties.get("eo:cloud_cover"),
            "geometry": shape(item.geometry),
        }
        for item in selected_items
    ],
    crs="EPSG:4326",
)

m_selected = leafmap.Map(
    center=[23.6850, 90.3563],
    zoom=7,
    height="700px",
)

m_selected.add_gdf(
    selected_scene_gdf,
    layer_name="Selected scenes",
    style={
        "color": "blue",
        "weight": 1,
        "fillColor": "blue",
        "fillOpacity": 0.08,
    },
)

m_selected.add_gdf(
    aoi,
    layer_name="Bangladesh boundary",
    style={"color": "red", "weight": 3, "fillOpacity": 0},
)

m_selected


In [ ]:
# 9. BUILD ONE RESILIENT SENTINEL-2 STACK
REQUIRED_ASSETS = ["blue", "green", "red", "nir", "swir16", "scl"]

selected_items_clean = [
    item
    for item in selected_items
    if all(asset in item.assets for asset in REQUIRED_ASSETS)
]

print("Scenes with all required assets:", len(selected_items_clean))
print(
    "Dropped for missing asset metadata:",
    len(selected_items) - len(selected_items_clean),
)

if not selected_items_clean:
    raise RuntimeError("No selected scene has all required bands.")

sentinel = stackstac.stack(
    selected_items_clean,
    assets=REQUIRED_ASSETS,
    bounds_latlon=[west, south, east, north],
    epsg=TARGET_EPSG,
    resolution=RESOLUTION_M,
    chunksize=CHUNK_SIZE,
    dtype=np.float32,
    fill_value=np.float32(np.nan),
    rescale=False,
    errors_as_nodata=(RasterioIOError(r".*"),),
)

print(sentinel)
print("Virtual stack shape:", sentinel.shape)


In [ ]:
# 10. SMALL REAL-READ TEST
cx = sentinel.sizes["x"] // 2
cy = sentinel.sizes["y"] // 2

test_stack = sentinel.isel(
    time=slice(0, min(3, sentinel.sizes["time"])),
    x=slice(max(0, cx - 128), min(cx + 128, sentinel.sizes["x"])),
    y=slice(max(0, cy - 128), min(cy + 128, sentinel.sizes["y"])),
).compute()

print("Small remote-read test successful.")
print("Shape:", test_stack.shape)
print("Finite values:", int(np.isfinite(test_stack.values).sum()))


In [ ]:
# 11. CLOUD MASK + TEMPORAL MEDIAN
scl = sentinel.sel(band="scl")

INVALID_SCL = [0, 1, 3, 8, 9, 10, 11]
valid = ~scl.isin(INVALID_SCL)

blue_med = sentinel.sel(band="blue").where(valid).median("time", skipna=True)
green_med = sentinel.sel(band="green").where(valid).median("time", skipna=True)
red_med = sentinel.sel(band="red").where(valid).median("time", skipna=True)
nir_med = sentinel.sel(band="nir").where(valid).median("time", skipna=True)
swir_med = sentinel.sel(band="swir16").where(valid).median("time", skipna=True)

print("Cloud-masked median composites prepared.")


In [ ]:
# 12. SENTINEL-2 INDICES
EPS = np.float32(1e-6)

def safe_nd(a, b):
    denominator = a + b
    return (
        (a - b) / denominator.where(np.abs(denominator) > EPS)
    ).clip(-1, 1).astype("float32")

ndbi = safe_nd(swir_med, nir_med).rename("NDBI")
ndvi = safe_nd(nir_med, red_med).rename("NDVI")
mndwi = safe_nd(green_med, swir_med).rename("MNDWI")

bsi_num = (swir_med + red_med) - (nir_med + blue_med)
bsi_den = (swir_med + red_med) + (nir_med + blue_med)

bsi = (
    bsi_num / bsi_den.where(np.abs(bsi_den) > EPS)
).clip(-1, 1).astype("float32").rename("BSI")

print("Prepared: NDBI, NDVI, MNDWI, BSI")


In [ ]:
# 13. BUILT-UP PROBABILITY (0–1)
# This is an interpretable score, not a pretrained classifier.

def index_to_01(da):
    return ((da + 1.0) / 2.0).clip(0, 1).astype("float32")

ndbi01 = index_to_01(ndbi)
ndvi01 = index_to_01(ndvi)
mndwi01 = index_to_01(mndwi)
bsi01 = index_to_01(bsi)

builtup_probability = (
    0.50 * ndbi01
    + 0.20 * (1.0 - ndvi01)
    + 0.20 * (1.0 - mndwi01)
    + 0.10 * (1.0 - bsi01)
).clip(0, 1).astype("float32").rename("Builtup_Probability")

print("Built-up probability READY.")


In [ ]:
# 14. CREATE COMMON 100-m TEMPLATE + EXACT BANGLADESH CLIP
template = (
    ndbi.astype("float32")
    .rio.set_spatial_dims(x_dim="x", y_dim="y", inplace=False)
    .rio.write_crs(f"EPSG:{TARGET_EPSG}", inplace=False)
)

aoi_target = aoi.to_crs(TARGET_EPSG)

def add_rio_metadata(da):
    return (
        da.rio
        .set_spatial_dims(x_dim="x", y_dim="y", inplace=False)
        .rio.write_crs(f"EPSG:{TARGET_EPSG}", inplace=False)
    )

def clip_bd(da):
    da = add_rio_metadata(da)
    return da.rio.clip(
        aoi_target.geometry,
        aoi_target.crs,
        drop=True,
        all_touched=False,
    )

def align_to_template(da, resampling="bilinear"):
    import rasterio.enums
    method = {
        "nearest": rasterio.enums.Resampling.nearest,
        "bilinear": rasterio.enums.Resampling.bilinear,
    }[resampling]

    return da.rio.reproject_match(
        template,
        resampling=method,
    ).astype("float32")

print("Template CRS:", template.rio.crs)
print("Template resolution:", template.rio.resolution())


## Part B — Online VIIRS, Population and Topography via Earth Engine + Xee

Xee lets Earth Engine imagery appear as lazy `xarray` objects in Python/VS Code. There is no manual raster download step.


In [ ]:
%pip install xee

In [ ]:
# 15. EARTH ENGINE AUTHENTICATION / INITIALIZATION

import ee

EE_PROJECT = "solid-garden-458417-t4"

try:
    ee.Initialize(project=EE_PROJECT)
    print("Earth Engine initialized successfully.")

except Exception:
    print("Authentication required...")
    ee.Authenticate()
    ee.Initialize(project=EE_PROJECT)
    print("Earth Engine initialized successfully after authentication.")

# Bangladesh geometry from your already-loaded shapefile
ee_aoi = ee.Geometry(bd_geom.__geo_interface__)

print("Bangladesh AOI transferred to Earth Engine.")
print("EE Project:", EE_PROJECT)

In [ ]:
# 16. HELPER: EARTH ENGINE IMAGE -> XARRAY ON 100-m BANGLADESH GRID

def ee_image_to_xarray(image, band_name, output_name):
    """
    Read one processed Earth Engine image lazily through Xee.
    The source is cloud-hosted; no manual input raster is needed.
    """
    image = ee.Image(image).select([band_name]).rename([output_name])

    # Fit the Bangladesh bounding box to a 100-m grid in EPSG:6933.
    grid = helpers.fit_geometry(
        geometry=bd_geom,
        geometry_crs="EPSG:4326",
        grid_crs=f"EPSG:{TARGET_EPSG}",
        grid_scale=(RESOLUTION_M, -RESOLUTION_M),
    )

    ds = xr.open_dataset(
        ee.ImageCollection([image]),
        engine="ee",
        **grid,
    )

    da = ds[output_name]

    if "time" in da.dims:
        da = da.isel(time=0, drop=True)

    # Xee uses x/y coordinates; attach CRS and match exact Sentinel grid.
    da = (
        da.astype("float32")
        .rio.set_spatial_dims(x_dim="x", y_dim="y", inplace=False)
        .rio.write_crs(f"EPSG:{TARGET_EPSG}", inplace=False)
    )

    return align_to_template(da, "bilinear")


def ee_percentile_norm(image, band, region, scale):
    """
    Robust 2nd–98th percentile normalization performed server-side.
    """
    image = ee.Image(image).select(band)

    stats = image.reduceRegion(
        reducer=ee.Reducer.percentile([2, 98]),
        geometry=region,
        scale=scale,
        bestEffort=True,
        maxPixels=1e9,
        tileScale=4,
    )

    lo = ee.Number(stats.get(f"{band}_p2"))
    hi = ee.Number(stats.get(f"{band}_p98"))

    return (
        image.subtract(lo)
        .divide(hi.subtract(lo).max(1e-6))
        .clamp(0, 1)
    )


In [ ]:
# 16. EARTH ENGINE IMAGE -> XARRAY
# No xee.helpers used

def ee_image_to_xarray(image, band_name, output_name):

    image = (
        ee.Image(image)
        .select([band_name])
        .rename([output_name])
    )

    collection = ee.ImageCollection([image])

    ds = xr.open_dataset(
        collection,
        engine="ee",
        geometry=bd_geom.__geo_interface__,
        crs=f"EPSG:{TARGET_EPSG}",
        scale=RESOLUTION_M,
    )

    da = ds[output_name]

    if "time" in da.dims:
        da = da.isel(time=0, drop=True)

    da = da.astype("float32")

    da = (
        da.rio
        .set_spatial_dims(
            x_dim="x",
            y_dim="y",
            inplace=False
        )
        .rio.write_crs(
            f"EPSG:{TARGET_EPSG}",
            inplace=False
        )
    )

    da = align_to_template(
        da,
        "bilinear"
    )

    return da


def ee_percentile_norm(
    image,
    band,
    region,
    scale
):

    image = ee.Image(image).select(band)

    stats = image.reduceRegion(
        reducer=ee.Reducer.percentile([2, 98]),
        geometry=region,
        scale=scale,
        bestEffort=True,
        maxPixels=1e9,
        tileScale=4,
    )

    lo = ee.Number(
        stats.get(f"{band}_p2")
    )

    hi = ee.Number(
        stats.get(f"{band}_p98")
    )

    normalized = (
        image
        .subtract(lo)
        .divide(
            hi.subtract(lo).max(1e-6)
        )
        .clamp(0, 1)
    )

    return normalized


print("EE helper functions updated successfully.")

In [ ]:
pip install --upgrade --pre xee

In [ ]:
import xee
print(xee.__version__)

In [ ]:
# ============================================================
# CELL 16 — CORRECT XEE GRID HELPER
# ============================================================

import ee
import xarray as xr
import rioxarray
from xee import helpers


def ee_image_to_xarray(image, band_name, output_name):

    image = (
        ee.Image(image)
        .select([band_name])
        .rename([output_name])
    )

    collection = ee.ImageCollection([image])

    # Build output grid for whole Bangladesh
    grid = helpers.fit_geometry(
        geometry=bd_geom,
        geometry_crs="EPSG:4326",
        grid_crs=f"EPSG:{TARGET_EPSG}",
        grid_scale=(RESOLUTION_M, -RESOLUTION_M),
    )

    ds = xr.open_dataset(
        collection,
        engine="ee",
        **grid,
    )

    da = ds[output_name]

    if "time" in da.dims:
        da = da.isel(time=0, drop=True)

    da = (
        da.astype("float32")
        .rio.set_spatial_dims(
            x_dim="x",
            y_dim="y",
            inplace=False,
        )
        .rio.write_crs(
            f"EPSG:{TARGET_EPSG}",
            inplace=False,
        )
    )

    # Align with Sentinel grid
    da = align_to_template(
        da,
        "bilinear",
    )

    return da


def ee_percentile_norm(
    image,
    band,
    region,
    scale,
):

    image = ee.Image(image).select(band)

    stats = image.reduceRegion(
        reducer=ee.Reducer.percentile([2, 98]),
        geometry=region,
        scale=scale,
        bestEffort=True,
        maxPixels=1e9,
        tileScale=4,
    )

    lo = ee.Number(
        stats.get(f"{band}_p2")
    )

    hi = ee.Number(
        stats.get(f"{band}_p98")
    )

    return (
        image
        .subtract(lo)
        .divide(
            hi.subtract(lo).max(1e-6)
        )
        .clamp(0, 1)
    )


print("Xee helper functions READY")

In [ ]:
# ============================================================
# VIIRS NIGHTTIME LIGHT — WORLD BANK OPEN NIGHT LIGHTS STYLE
# BANGLADESH 2025
# ============================================================

import ee
import geemap


# ------------------------------------------------------------
# 1. EARTH ENGINE INITIALIZATION
# ------------------------------------------------------------

EE_PROJECT = "solid-garden-458417-t4"

try:
    ee.Initialize(project=EE_PROJECT)
    print("Earth Engine initialized.")

except Exception:

    ee.Authenticate()

    ee.Initialize(project=EE_PROJECT)

    print("Earth Engine initialized after authentication.")


# ------------------------------------------------------------
# 2. BANGLADESH AOI
# ------------------------------------------------------------
# bd_geom already comes from your Bangladesh shapefile

ee_aoi = ee.Geometry(
    bd_geom.__geo_interface__
)

print("Bangladesh AOI ready.")


# ------------------------------------------------------------
# 3. VIIRS MONTHLY COLLECTION — 2025
# ------------------------------------------------------------

viirs_2025 = (
    ee.ImageCollection(
        "NOAA/VIIRS/DNB/MONTHLY_V1/VCMSLCFG"
    )

    .filterDate(
        "2025-01-01",
        "2026-01-01"
    )

    .filterBounds(
        ee_aoi
    )

    .select(
        "avg_rad"
    )
)


print(
    "VIIRS monthly images:",
    viirs_2025.size().getInfo()
)


# ------------------------------------------------------------
# 4. CLIP EVERY MONTH TO BANGLADESH
# ------------------------------------------------------------

viirs_bd = viirs_2025.map(
    lambda image:
        image
        .clip(ee_aoi)
        .copyProperties(
            image,
            ["system:time_start"]
        )
)


# ------------------------------------------------------------
# 5. ANNUAL MEDIAN COMPOSITE
# ------------------------------------------------------------
# Same basic logic as World Bank Open Night Lights tutorial

viirs_annual = (
    viirs_bd
    .median()
    .rename(
        "VIIRS_2025"
    )
)


print(
    "Annual VIIRS composite READY."
)


# ------------------------------------------------------------
# 6. MASK ZERO / NEGATIVE VALUES
# ------------------------------------------------------------

viirs_annual = (
    viirs_annual
    .updateMask(
        viirs_annual.gt(0)
    )
)


# ------------------------------------------------------------
# 7. VISUALIZATION PARAMETERS
# ------------------------------------------------------------

viirs_vis = {
    "min": 0,
    "max": 20,
    "palette": [
        "000000",
        "0d0887",
        "7e03a8",
        "cc4778",
        "f89540",
        "f0f921"
    ]
}


# ------------------------------------------------------------
# 8. SHOW IN VS CODE
# ------------------------------------------------------------

Map = geemap.Map(
    center=[
        23.6850,
        90.3563
    ],
    zoom=7,
    height="750px"
)


Map.add_basemap(
    "SATELLITE"
)


Map.addLayer(
    viirs_annual,
    viirs_vis,
    "VIIRS-DNB Bangladesh 2025"
)


# Bangladesh boundary
boundary = ee.Image().byte().paint(
    featureCollection=ee.FeatureCollection(
        [
            ee.Feature(
                ee_aoi
            )
        ]
    ),
    color=1,
    width=2
)


Map.addLayer(
    boundary,
    {
        "palette": ["red"]
    },
    "Bangladesh Boundary"
)


Map.addLayerControl()

Map

In [ ]:
# ============================================================
# GHSL POPULATION 2025 — EARTH ENGINE NATIVE
# NO XEE
# ============================================================

import ee
import geemap


# ------------------------------------------------------------
# 1. LOAD GHSL 2025
# ------------------------------------------------------------

ghsl_col = (
    ee.ImageCollection("JRC/GHSL/P2023A/GHS_POP")
    .filterDate("2025-01-01", "2026-01-01")
    .filterBounds(ee_aoi)
)

ghsl_n = ghsl_col.size().getInfo()

print("GHSL 2025 images:", ghsl_n)

if ghsl_n == 0:
    raise RuntimeError(
        "No GHSL population image found for 2025."
    )


# ------------------------------------------------------------
# 2. POPULATION COUNT
# ------------------------------------------------------------

pop_count_ee = (
    ee.Image(ghsl_col.first())
    .select("population_count")
    .max(0)
    .clip(ee_aoi)
    .rename("population_count")
)


# ------------------------------------------------------------
# 3. POPULATION DENSITY
# ------------------------------------------------------------

pop_density_ee = (
    pop_count_ee
    .multiply(100.0)
    .rename("population_density")
)


# ------------------------------------------------------------
# 4. LOG TRANSFORMATION
# ------------------------------------------------------------

pop_log = (
    pop_density_ee
    .add(1)
    .log()
    .rename("pop_log")
)


# ------------------------------------------------------------
# 5. 0–1 POPULATION SCORE
# ------------------------------------------------------------

population_score_ee = (
    ee_percentile_norm(
        pop_log,
        "pop_log",
        ee_aoi,
        100
    )
    .rename("Population_Score")
    .clip(ee_aoi)
)


# ------------------------------------------------------------
# 6. CHECK
# ------------------------------------------------------------

print("population : READY")

print(
    "Population bands:",
    population_score_ee.bandNames().getInfo()
)

In [ ]:
# ============================================================
# SHOW GHSL POPULATION SCORE
# ============================================================

pop_vis = {
    "min": 0,
    "max": 1,
    "palette": [
        "ffffcc",
        "c2e699",
        "78c679",
        "31a354",
        "006837"
    ]
}

Map_pop = geemap.Map(
    center=[23.6850, 90.3563],
    zoom=7,
    height="750px"
)

Map_pop.add_basemap("SATELLITE")

Map_pop.addLayer(
    population_score_ee,
    pop_vis,
    "GHSL Population Score 2025"
)

Map_pop.addLayer(
    ee.Image().byte().paint(
        ee.FeatureCollection([
            ee.Feature(ee_aoi)
        ]),
        1,
        2
    ),
    {"palette": ["red"]},
    "Bangladesh Boundary"
)

Map_pop.addLayerControl()

Map_pop

In [ ]:
# ============================================================
# 21. COPERNICUS DEM + SLOPE — ONLINE
# EARTH ENGINE NATIVE
# NO XEE
# ============================================================

import ee
import geemap


# ------------------------------------------------------------
# 1. LOAD COPERNICUS DEM
# ------------------------------------------------------------

dem_col = ee.ImageCollection(
    "COPERNICUS/DEM/GLO30_2024_1"
)

native_projection = (
    dem_col
    .first()
    .select("DEM")
    .projection()
)


# ------------------------------------------------------------
# 2. MOSAIC DEM
# ------------------------------------------------------------

dem_ee = (
    dem_col
    .select("DEM")
    .mosaic()
    .setDefaultProjection(
        native_projection
    )
    .clip(
        ee_aoi
    )
    .rename(
        "elevation"
    )
)


print(
    "DEM ready."
)


# ------------------------------------------------------------
# 3. SLOPE
# ------------------------------------------------------------

slope_ee = (
    ee.Terrain
    .slope(
        dem_ee
    )
    .clip(
        ee_aoi
    )
    .rename(
        "slope"
    )
)


print(
    "Slope ready."
)


# ------------------------------------------------------------
# 4. NORMALIZE ELEVATION
# ------------------------------------------------------------

elev01_ee = (
    ee_percentile_norm(
        dem_ee,
        "elevation",
        ee_aoi,
        100
    )
    .rename(
        "elevation01"
    )
)


# ------------------------------------------------------------
# 5. NORMALIZE SLOPE
# ------------------------------------------------------------

slope01_ee = (
    ee_percentile_norm(
        slope_ee,
        "slope",
        ee_aoi,
        100
    )
    .rename(
        "slope01"
    )
)


# ------------------------------------------------------------
# 6. TOPOGRAPHY SCORE
# ------------------------------------------------------------
#
# Lower elevation = higher score
# Lower slope     = higher score
#
# Weight:
# elevation = 0.50
# slope     = 0.50
#

topography_ee = (
    ee.Image(1)
    .subtract(
        elev01_ee
    )
    .multiply(
        0.50
    )

    .add(

        ee.Image(1)
        .subtract(
            slope01_ee
        )
        .multiply(
            0.50
        )

    )

    .clamp(
        0,
        1
    )

    .clip(
        ee_aoi
    )

    .rename(
        "Topography_Score"
    )
)


# ------------------------------------------------------------
# 7. CHECK
# ------------------------------------------------------------

print(
    "Topography bands:",
    topography_ee.bandNames().getInfo()
)

print()
print(
    "topography  : READY"
)

In [ ]:
# ============================================================
# 22. SHOW DEM / SLOPE / TOPOGRAPHY SCORE
# ============================================================

Map_topo = geemap.Map(
    center=[
        23.6850,
        90.3563
    ],
    zoom=7,
    height="750px"
)


# ------------------------------------------------------------
# DEM
# ------------------------------------------------------------

dem_vis = {
    "min": 0,
    "max": 100,
    "palette": [
        "006837",
        "31a354",
        "78c679",
        "c2e699",
        "ffffcc",
        "fed976",
        "fd8d3c",
        "bd0026"
    ]
}


Map_topo.addLayer(
    dem_ee,
    dem_vis,
    "Elevation"
)


# ------------------------------------------------------------
# SLOPE
# ------------------------------------------------------------

slope_vis = {
    "min": 0,
    "max": 30,
    "palette": [
        "ffffcc",
        "a1dab4",
        "41b6c4",
        "2c7fb8",
        "253494"
    ]
}


Map_topo.addLayer(
    slope_ee,
    slope_vis,
    "Slope"
)


# ------------------------------------------------------------
# TOPOGRAPHY SCORE
# ------------------------------------------------------------

topo_vis = {
    "min": 0,
    "max": 1,
    "palette": [
        "d73027",
        "fc8d59",
        "fee08b",
        "d9ef8b",
        "91cf60",
        "1a9850"
    ]
}


Map_topo.addLayer(
    topography_ee,
    topo_vis,
    "Topography Score"
)


# ------------------------------------------------------------
# BANGLADESH BOUNDARY
# ------------------------------------------------------------

boundary = (
    ee.Image()
    .byte()
    .paint(
        ee.FeatureCollection(
            [
                ee.Feature(
                    ee_aoi
                )
            ]
        ),
        1,
        2
    )
)


Map_topo.addLayer(
    boundary,
    {
        "palette": [
            "red"
        ]
    },
    "Bangladesh Boundary"
)


Map_topo.addLayerControl()

Map_topo


## Part C — Roads + POI directly from Overture Maps cloud

DuckDB reads the public Overture GeoParquet files directly from Amazon S3 and applies the Bangladesh bounding-box filter in the cloud query. You do not have to manually download a Bangladesh roads or POI file.


In [ ]:
# 23. CONNECT DUCKDB TO OVERTURE CLOUD
con = duckdb.connect(database=":memory:")

# DuckDB extensions.
con.execute("INSTALL spatial")
con.execute("LOAD spatial")
con.execute("INSTALL httpfs")
con.execute("LOAD httpfs")

con.execute("SET s3_region='us-west-2'")
con.execute("SET enable_object_cache=true")
con.execute("SET threads=4")

ROAD_URL = (
    "s3://overturemaps-us-west-2/release/"
    f"{OVERTURE_RELEASE}/theme=transportation/type=segment/*"
)

POI_URL = (
    "s3://overturemaps-us-west-2/release/"
    f"{OVERTURE_RELEASE}/theme=places/type=place/*"
)

print("DuckDB + Overture cloud READY")


In [ ]:
import duckdb
import geopandas as gpd
import numpy as np
import xarray as xr

from shapely import from_wkb
from shapely.geometry import Point

from rasterio.features import rasterize
from rasterio.enums import MergeAlg
from scipy import ndimage

OVERTURE_RELEASE = "2026-08-19.0"

ROAD_URL = (
    "s3://overturemaps-us-west-2/release/"
    f"{OVERTURE_RELEASE}/theme=transportation/type=segment/*"
)

print("Overture release:", OVERTURE_RELEASE)
print("ROAD_URL:", ROAD_URL)

In [ ]:
# STEP 2 — DUCKDB CONNECTION

import duckdb

con = duckdb.connect(database=":memory:")

con.execute("INSTALL spatial")
con.execute("LOAD spatial")

con.execute("INSTALL httpfs")
con.execute("LOAD httpfs")

con.execute("SET s3_region='us-west-2'")
con.execute("SET enable_object_cache=true")
con.execute("SET threads=4")

print("DuckDB ready.")

In [ ]:
# STEP 3 — CHECK BANGLADESH AOI VARIABLES

print("west :", west)
print("south:", south)
print("east :", east)
print("north:", north)

print("AOI CRS:", aoi.crs)

In [ ]:
# STEP 4 — QUERY BANGLADESH ROADS ONLINE

road_sql = f"""
SELECT
    id,
    subtype,
    class,
    ST_AsWKB(geometry) AS geom_wkb

FROM read_parquet(
    '{ROAD_URL}',
    hive_partitioning=1
)

WHERE
    subtype = 'road'

    AND class IN (
        'motorway',
        'trunk',
        'primary',
        'secondary',
        'tertiary',
        'residential',
        'living_street',
        'unclassified',
        'service'
    )

    AND bbox.xmin <= {east}
    AND bbox.xmax >= {west}
    AND bbox.ymin <= {north}
    AND bbox.ymax >= {south}
"""

print("Querying Overture roads online...")

roads_df = con.execute(
    road_sql
).fetch_df()

print("Raw road records:", len(roads_df))

In [ ]:
# ============================================================
# STEP 5 — CONVERT OVERTURE ROADS TO GEODATAFRAME
# FIX: bytearray -> bytes
# ============================================================

import geopandas as gpd
from shapely import from_wkb

print("Converting road geometries...")

if roads_df.empty:
    raise RuntimeError("roads_df is empty.")


# ------------------------------------------------------------
# 1. bytearray -> bytes
# ------------------------------------------------------------

wkb_values = roads_df["geom_wkb"].apply(
    lambda x: bytes(x) if isinstance(x, bytearray) else x
)


# ------------------------------------------------------------
# 2. WKB -> Shapely geometry
# ------------------------------------------------------------

road_geometry = from_wkb(
    wkb_values.to_numpy()
)


# ------------------------------------------------------------
# 3. GeoDataFrame
# ------------------------------------------------------------

roads = gpd.GeoDataFrame(
    roads_df.drop(
        columns=["geom_wkb"]
    ),
    geometry=road_geometry,
    crs="EPSG:4326"
)


# ------------------------------------------------------------
# 4. Remove null / empty geometry
# ------------------------------------------------------------

roads = roads[
    roads.geometry.notna()
    &
    ~roads.geometry.is_empty
].copy()


# ------------------------------------------------------------
# 5. Check
# ------------------------------------------------------------

print()
print("=" * 55)
print("Road GeoDataFrame : READY")
print("Road segments     :", len(roads))
print("CRS               :", roads.crs)
print("=" * 55)

display(
    roads.head()
)

In [ ]:
# ============================================================
# STEP 6 — FAST BANGLADESH ROAD FILTER
# DO NOT USE gpd.clip() ON 1.7 MILLION FEATURES
# ============================================================

print("Road segments before filter:", len(roads))

# Bangladesh polygon
bd_polygon = aoi.geometry.union_all()

print("Filtering roads by Bangladesh intersection...")

# Spatial-index-assisted filtering
roads_bd = roads[
    roads.geometry.intersects(bd_polygon)
].copy()

# Remove empty geometries
roads_bd = roads_bd[
    roads_bd.geometry.notna()
    &
    ~roads_bd.geometry.is_empty
].copy()

print()
print("=" * 55)
print("BANGLADESH ROAD FILTER : READY")
print("Before :", len(roads))
print("After  :", len(roads_bd))
print("CRS    :", roads_bd.crs)
print("=" * 55)

display(roads_bd.head())

In [ ]:
# 25. SHOW OVERTURE ROADS IN VS CODE
#
# For notebook responsiveness, display only a sample if there are many roads.

roads_display = roads

if len(roads_display) > 50000:
    roads_display = roads_display.sample(
        50000,
        random_state=42,
    )

m_roads = leafmap.Map(
    center=[23.6850, 90.3563],
    zoom=7,
    height="700px",
)

m_roads.add_gdf(
    roads_display,
    layer_name="Overture Roads (display sample)",
    style={"color": "blue", "weight": 1},
)

m_roads.add_gdf(
    aoi,
    layer_name="Bangladesh",
    style={"color": "red", "weight": 2, "fillOpacity": 0},
)

m_roads


In [ ]:
# 26. ROAD DENSITY + INTERSECTION DENSITY -> ROAD SCORE

roads_m = roads.to_crs(TARGET_EPSG)

# Remove empty geometry.
roads_m = roads_m[
    roads_m.geometry.notna()
    & ~roads_m.geometry.is_empty
].copy()

transform = template.rio.transform()
out_shape = (
    template.sizes["y"],
    template.sizes["x"],
)

# 1) Road-presence raster
road_presence = rasterize(
    [(geom, 1) for geom in roads_m.geometry],
    out_shape=out_shape,
    transform=transform,
    fill=0,
    dtype="uint8",
    all_touched=True,
)

# Approx. 1 km x 1 km window at 100 m.
window_cells = max(
    1,
    int(round(1000 / RESOLUTION_M)),
)

road_density_np = ndimage.uniform_filter(
    road_presence.astype("float32"),
    size=window_cells,
    mode="constant",
    cval=0,
)

# 2) Intersection proxy:
# Count repeated road endpoints after rounding projected coordinates.
endpoint_keys = []

for geom in roads_m.geometry:
    try:
        coords = list(geom.coords)
        if len(coords) >= 2:
            for xy in (coords[0], coords[-1]):
                endpoint_keys.append(
                    (
                        round(xy[0], 0),
                        round(xy[1], 0),
                    )
                )
    except Exception:
        pass

endpoint_counts = Counter(endpoint_keys)

intersection_points = [
    Point(x, y)
    for (x, y), count in endpoint_counts.items()
    if count >= 3
]

intersection_raster = rasterize(
    [(geom, 1) for geom in intersection_points],
    out_shape=out_shape,
    transform=transform,
    fill=0,
    dtype="float32",
    merge_alg=MergeAlg.add,
)

intersection_density_np = ndimage.uniform_filter(
    intersection_raster,
    size=window_cells,
    mode="constant",
    cval=0,
)

def np_to_template(arr, name):
    da = xr.DataArray(
        arr.astype("float32"),
        dims=("y", "x"),
        coords={
            "y": template.y.values,
            "x": template.x.values,
        },
        name=name,
    )
    return add_rio_metadata(da)

road_density = np_to_template(
    road_density_np,
    "Road_Density",
)

intersection_density = np_to_template(
    intersection_density_np,
    "Intersection_Density",
)

def robust_norm_local(da):
    arr = da.values
    vals = arr[np.isfinite(arr)]

    if vals.size == 0:
        raise ValueError("No finite data for normalization.")

    lo, hi = np.nanpercentile(vals, [2, 98])

    if hi <= lo:
        return xr.zeros_like(da, dtype="float32")

    return (
        (da - lo) / (hi - lo)
    ).clip(0, 1).astype("float32")

road_density01 = robust_norm_local(road_density)
intersection_density01 = robust_norm_local(intersection_density)

road_score = (
    0.70 * road_density01
    + 0.30 * intersection_density01
).clip(0, 1).astype("float32").rename("Road_Score")

print("Intersections:", len(intersection_points))
print("road        : READY")


In [ ]:
# ============================================================
# FIX OVERTURE RELEASE + URLS
# ============================================================

OVERTURE_RELEASE = "2026-08-19.0"

ROAD_URL = (
    f"s3://overturemaps-us-west-2/release/"
    f"{OVERTURE_RELEASE}/"
    f"theme=transportation/type=segment/*"
)

POI_URL = (
    f"s3://overturemaps-us-west-2/release/"
    f"{OVERTURE_RELEASE}/"
    f"theme=places/type=place/*"
)

print("Overture release:", OVERTURE_RELEASE)
print("ROAD_URL:", ROAD_URL)
print("POI_URL :", POI_URL)

In [ ]:
print(POI_URL)

In [ ]:
# ============================================================
# CELL 27 — FIXED OVERTURE POI / URBAN SERVICES
# BANGLADESH — ALL ONLINE
# ============================================================

import numpy as np
import geopandas as gpd
from shapely import from_wkb


# ------------------------------------------------------------
# 1. CHECK REQUIRED VARIABLES
# ------------------------------------------------------------

required = [
    "con",
    "POI_URL",
    "west",
    "south",
    "east",
    "north",
    "aoi",
]

missing = [
    name for name in required
    if name not in globals()
]

if missing:
    raise RuntimeError(
        "Required variables are missing:\n"
        f"{missing}\n\n"
        "Run Master Initialization and DuckDB setup first."
    )


print("POI URL:")
print(POI_URL)


# ------------------------------------------------------------
# 2. TEST OVERTURE POI DATASET
# ------------------------------------------------------------

print("\nTesting Overture Places dataset...")

test_sql = f"""
SELECT COUNT(*) AS n
FROM read_parquet(
    '{POI_URL}',
    hive_partitioning=1
)
WHERE
    bbox.xmin <= {east}
    AND bbox.xmax >= {west}
    AND bbox.ymin <= {north}
    AND bbox.ymax >= {south}
"""

try:

    poi_test = con.execute(
        test_sql
    ).fetch_df()

    print(
        "POIs intersecting Bangladesh bbox:",
        int(poi_test.iloc[0]["n"])
    )

except Exception as exc:

    raise RuntimeError(
        "\nOverture POI dataset could not be opened.\n"
        "Most likely POI_URL / Overture release is invalid.\n\n"
        f"Current POI_URL:\n{POI_URL}\n\n"
        f"Original error:\n{exc}"
    )


# ------------------------------------------------------------
# 3. QUERY URBAN-SERVICE POIs
# ------------------------------------------------------------

poi_sql = f"""
SELECT
    id,
    basic_category,
    CAST(taxonomy AS VARCHAR) AS taxonomy_text,
    ST_AsWKB(geometry) AS geom_wkb

FROM read_parquet(
    '{POI_URL}',
    hive_partitioning=1
)

WHERE

    bbox.xmin <= {east}
    AND bbox.xmax >= {west}
    AND bbox.ymin <= {north}
    AND bbox.ymax >= {south}

    AND (

        regexp_matches(
            lower(
                coalesce(
                    basic_category,
                    ''
                )
            ),
            'market|shop|mall|grocery|supermarket|school|college|university|hospital|clinic|medical|bank|office|restaurant|hotel|pharmacy|service'
        )

        OR

        regexp_matches(
            lower(
                coalesce(
                    CAST(taxonomy AS VARCHAR),
                    ''
                )
            ),
            'market|shopping|school|education|college|university|hospital|clinic|health|medical|commercial|business|bank|office|restaurant|hotel|pharmacy|service'
        )

    )
"""


print(
    "\nQuerying urban-service POIs..."
)


try:

    poi_df = con.execute(
        poi_sql
    ).fetch_df()

except Exception as exc:

    raise RuntimeError(
        "Overture POI query failed.\n\n"
        f"{exc}"
    )


print(
    "Raw urban-service records:",
    len(poi_df)
)


if poi_df.empty:

    raise RuntimeError(
        "Overture query worked, but no "
        "urban-service POIs were returned."
    )


# ------------------------------------------------------------
# 4. FIX WKB BYTEARRAY PROBLEM
# ------------------------------------------------------------

print(
    "Converting POI geometries..."
)


def wkb_to_bytes(value):

    if value is None:
        return None

    if isinstance(
        value,
        bytearray
    ):
        return bytes(value)

    if isinstance(
        value,
        memoryview
    ):
        return value.tobytes()

    return value


wkb_values = np.array(
    [
        wkb_to_bytes(value)
        for value in poi_df["geom_wkb"]
    ],
    dtype=object,
)


poi_geometry = from_wkb(
    wkb_values,
    on_invalid="ignore",
)


# ------------------------------------------------------------
# 5. CREATE GEODATAFRAME
# ------------------------------------------------------------

pois = gpd.GeoDataFrame(

    poi_df.drop(
        columns=["geom_wkb"]
    ),

    geometry=poi_geometry,

    crs="EPSG:4326",
)


# ------------------------------------------------------------
# 6. REMOVE INVALID / EMPTY GEOMETRY
# ------------------------------------------------------------

pois = pois[
    pois.geometry.notna()
].copy()

pois = pois[
    ~pois.geometry.is_empty
].copy()


print(
    "Valid POI geometries:",
    len(pois)
)


# ------------------------------------------------------------
# 7. EXACT BANGLADESH FILTER
# ------------------------------------------------------------
# Spatial predicate is enough for point POIs.
# Avoid expensive gpd.clip().

print(
    "Filtering exact Bangladesh boundary..."
)


bd_union = aoi.geometry.union_all()


pois = pois[
    pois.geometry.intersects(
        bd_union
    )
].copy()


# ------------------------------------------------------------
# 8. REMOVE DUPLICATES
# ------------------------------------------------------------

pois = pois.drop_duplicates(
    subset=["id"]
).reset_index(
    drop=True
)


# ------------------------------------------------------------
# 9. FINAL CHECK
# ------------------------------------------------------------

if pois.empty:

    raise RuntimeError(
        "No POIs remained after Bangladesh boundary filtering."
    )


print()
print("=" * 65)

print(
    "OVERTURE POI QUERY SUCCESSFUL"
)

print("=" * 65)

print(
    "Urban-service POIs:",
    len(pois)
)

print(
    "CRS:",
    pois.crs
)

print()

print(
    pois[
        [
            "id",
            "basic_category",
            "taxonomy_text"
        ]
    ].head(10)
)

print("=" * 65)

print(
    "poi : READY"
)

print("=" * 65)

In [ ]:
print("Querying Overture POIs online...")
poi_df = con.execute(poi_sql).fetch_df()

if poi_df.empty:
    raise RuntimeError("Overture POI query returned no records.")

print("POIs returned from Overture:", len(poi_df))

# ------------------------------------------------------------
# FIX: DuckDB WKB bytearray -> Python bytes
# ------------------------------------------------------------
geom_wkb = poi_df.pop("geom_wkb")

poi_geoms = from_wkb(
    np.array(
        [
            bytes(g) if isinstance(g, (bytearray, memoryview)) else g
            for g in geom_wkb
        ],
        dtype=object,
    )
)

# ------------------------------------------------------------
# Create GeoDataFrame
# ------------------------------------------------------------
pois = gpd.GeoDataFrame(
    poi_df,
    geometry=poi_geoms,
    crs="EPSG:4326",
)

# Remove invalid / empty geometries
pois = pois[
    pois.geometry.notna()
    & ~pois.geometry.is_empty
].copy()

# ------------------------------------------------------------
# Clip exactly to Bangladesh AOI
# ------------------------------------------------------------
pois = gpd.clip(pois, aoi)

pois = pois.reset_index(drop=True)

print("Urban-service POIs in Bangladesh:", len(pois))
print()
print(pois.head())

print("\nGeometry types:")
print(pois.geometry.geom_type.value_counts())

In [ ]:
# 28. CLASSIFY POIs INTO THE WORKFLOW'S FOUR GROUPS

def classify_poi(row):
    text = (
        str(row.get("basic_category", ""))
        + " "
        + str(row.get("taxonomy_text", ""))
    ).lower()

    if any(k in text for k in [
        "school", "college", "university", "education",
        "academy", "kindergarten",
    ]):
        return "education"

    if any(k in text for k in [
        "hospital", "clinic", "medical", "health",
        "pharmacy", "doctor",
    ]):
        return "health"

    if any(k in text for k in [
        "market", "supermarket", "grocery", "mall",
        "shopping", "shop",
    ]):
        return "market"

    return "commercial_service"


pois["poi_group"] = pois.apply(
    classify_poi,
    axis=1,
)

print(pois["poi_group"].value_counts())


In [ ]:
# 29. POI DENSITIES -> POI/SERVICE SCORE

pois_m = pois.to_crs(TARGET_EPSG)

poi_group_scores = []

for group in [
    "market",
    "education",
    "health",
    "commercial_service",
]:
    subset = pois_m[pois_m["poi_group"] == group]

    group_raster = rasterize(
        [(geom, 1) for geom in subset.geometry],
        out_shape=out_shape,
        transform=transform,
        fill=0,
        dtype="float32",
        merge_alg=MergeAlg.add,
    )

    group_density_np = ndimage.uniform_filter(
        group_raster,
        size=window_cells,
        mode="constant",
        cval=0,
    )

    group_da = np_to_template(
        group_density_np,
        f"{group}_density",
    )

    group_score = robust_norm_local(group_da)
    poi_group_scores.append(group_score)

poi_score = (
    sum(poi_group_scores) / len(poi_group_scores)
).clip(0, 1).astype("float32").rename("POI_Service_Score")

print("poi         : READY")


In [ ]:
# ============================================================
# 29. CREATE NIGHTLIGHT SCORE FROM VIIRS ANNUAL COMPOSITE
# ============================================================

print("Creating nightlight score...")

# Use a practical upper cap so extreme bright pixels
# do not dominate the whole Bangladesh score.
VIIRS_MAX = 20.0

nightlight_score = (
    viirs_annual
    .divide(VIIRS_MAX)
    .clamp(0, 1)
    .rename("nightlight_score")
)

# Keep only valid illuminated pixels
nightlight_score = nightlight_score.updateMask(
    viirs_annual.mask()
)

print("nightlight_score READY.")

In [ ]:
nightlight_score_vis = {
    "min": 0,
    "max": 1,
    "palette": [
        "000000",
        "313695",
        "74add1",
        "ffffbf",
        "f46d43",
        "a50026"
    ]
}

Map2 = geemap.Map(
    center=[23.6850, 90.3563],
    zoom=7,
    height="700px"
)

Map2.addLayer(
    nightlight_score,
    nightlight_score_vis,
    "Nightlight Score 0-1"
)

Map2.addLayerControl()

Map2

In [ ]:
# ============================================================
# POPULATION SCORE — GHSL 2025
# CORRECTED VERSION
# ============================================================

print("Loading GHSL 2025 population...")

# ------------------------------------------------------------
# 1. Load 2025 population image directly
# ------------------------------------------------------------

population = (
    ee.Image("JRC/GHSL/P2023A/GHS_POP/2025")
    .select("population_count")
    .clip(ee_aoi)
)

print("Population bands:", population.bandNames().getInfo())


# ------------------------------------------------------------
# 2. Remove zero / negative population
# ------------------------------------------------------------

population = population.updateMask(
    population.gt(0)
)


# ------------------------------------------------------------
# 3. Log-transform
# ------------------------------------------------------------
# Population distribution is highly skewed.
# log(1 + population) reduces extreme urban values.

population_log = (
    population
    .add(1)
    .log()
    .rename("population_log")
)


# ------------------------------------------------------------
# 4. Calculate Bangladesh P2 and P98
# ------------------------------------------------------------

pop_stats = population_log.reduceRegion(
    reducer=ee.Reducer.percentile([2, 98]),
    geometry=ee_aoi,
    scale=1000,
    bestEffort=True,
    maxPixels=1e13,
    tileScale=4
).getInfo()

print("Population percentile statistics:")
print(pop_stats)


# ------------------------------------------------------------
# 5. Extract percentile values
# ------------------------------------------------------------

p2 = pop_stats["population_log_p2"]
p98 = pop_stats["population_log_p98"]

print("P2 :", p2)
print("P98:", p98)


# ------------------------------------------------------------
# 6. Normalize to 0–1
# ------------------------------------------------------------

population_score = (
    population_log
    .subtract(p2)
    .divide(p98 - p2)
    .clamp(0, 1)
    .rename("population_score")
)


print("population_score READY.")

In [ ]:
print(
    "population_score bands:",
    population_score.bandNames().getInfo()
)

print(
    "population_score type:",
    type(population_score)
)

In [ ]:
# ============================================================
# TOPOGRAPHY SCORE — COPERNICUS DEM GLO30
# BANGLADESH
# ============================================================

print("Creating topography score...")

# ------------------------------------------------------------
# 1. Load Copernicus DEM
# ------------------------------------------------------------

dem = (
    ee.ImageCollection("COPERNICUS/DEM/GLO30")
    .filterBounds(ee_aoi)
    .select("DEM")
    .mosaic()
    .clip(ee_aoi)
)

print("DEM bands:", dem.bandNames().getInfo())


# ------------------------------------------------------------
# 2. Calculate slope
# ------------------------------------------------------------

slope = (
    ee.Terrain.slope(dem)
    .rename("slope")
)

print("Slope READY.")


# ------------------------------------------------------------
# 3. Calculate Bangladesh slope percentiles
# ------------------------------------------------------------

slope_stats = slope.reduceRegion(
    reducer=ee.Reducer.percentile([2, 98]),
    geometry=ee_aoi,
    scale=1000,
    bestEffort=True,
    maxPixels=1e13,
    tileScale=4
).getInfo()

print("Slope percentile statistics:")
print(slope_stats)


# ------------------------------------------------------------
# 4. Extract percentile values
# ------------------------------------------------------------

slope_p2 = slope_stats["slope_p2"]
slope_p98 = slope_stats["slope_p98"]

print("Slope P2 :", slope_p2)
print("Slope P98:", slope_p98)


# ------------------------------------------------------------
# 5. Normalize slope to 0–1
# ------------------------------------------------------------
# High slope -> high normalized value

slope_norm = (
    slope
    .subtract(slope_p2)
    .divide(slope_p98 - slope_p2)
    .clamp(0, 1)
)


# ------------------------------------------------------------
# 6. Invert for urban suitability
# ------------------------------------------------------------
# Flat area      -> score close to 1
# Steep terrain  -> score close to 0

topography_score = (
    ee.Image.constant(1)
    .subtract(slope_norm)
    .rename("topography_score")
    .clip(ee_aoi)
)

print("topography_score READY.")


# ------------------------------------------------------------
# 7. Verify
# ------------------------------------------------------------

print(
    "topography_score bands:",
    topography_score.bandNames().getInfo()
)

print(
    "topography_score type:",
    type(topography_score)
)

In [ ]:
# ============================================================
# 30. SAFE FACTOR READINESS CHECK
# ============================================================

factor_names = [
    "builtup_probability",
    "nightlight_score",
    "population_score",
    "road_score",
    "poi_score",
    "topography_score",
]

factor_labels = [
    "builtup",
    "nightlight",
    "population",
    "road",
    "poi",
    "topography",
]

print("FACTOR STATUS")
print("-" * 32)

missing = []

for label, var_name in zip(factor_labels, factor_names):

    if var_name in globals():
        value = globals()[var_name]

        if value is not None:
            print(f"{label:12s}: READY")
        else:
            print(f"{label:12s}: MISSING")
            missing.append(label)

    else:
        print(f"{label:12s}: NOT DEFINED")
        missing.append(label)


if missing:
    print("\nMissing factors:", ", ".join(missing))

else:
    print("\nALL SIX FACTORS ARE READY.")

In [ ]:
# ============================================================
# 30B. CREATE FACTORS DICTIONARY
# ============================================================

factors = {
    "builtup": builtup_probability,
    "nightlight": nightlight_score,
    "population": population_score,
    "road": road_score,
    "poi": poi_score,
    "topography": topography_score,
}

print("FACTORS DICTIONARY CREATED")
print("-" * 60)

for name, value in factors.items():
    print(f"{name:12s}: {type(value)}")

In [ ]:
# ============================================================
# 31. SHOW ALL SIX FACTORS — MIXED TYPE SAFE
# ============================================================

import matplotlib.pyplot as plt
import ee
import geemap

factor_titles = {
    "builtup": "Built-up Probability",
    "nightlight": "Night-Light Score",
    "population": "Population Score",
    "road": "Road Score",
    "poi": "POI / Service Score",
    "topography": "Topography Score",
}

# ------------------------------------------------------------
# A. LOCAL / XARRAY FACTORS
# ------------------------------------------------------------

for name, da in factors.items():

    # Skip Earth Engine images here
    if isinstance(da, ee.image.Image):
        continue

    print(f"Showing local factor: {name}")

    try:
        preview = clip_bd(da)

        # Downsample only if this is an xarray-like object
        if hasattr(preview, "coarsen"):
            preview = (
                preview
                .coarsen(x=5, y=5, boundary="trim")
                .mean(skipna=True)
            )

        if hasattr(preview, "compute"):
            preview = preview.compute()

        plt.figure(figsize=(7, 8))

        preview.plot(
            vmin=0,
            vmax=1
        )

        plt.title(
            factor_titles[name],
            fontsize=14
        )

        plt.axis("equal")
        plt.tight_layout()
        plt.show()

    except Exception as exc:

        print(
            f"Could not display {name}: {exc}"
        )


# ------------------------------------------------------------
# B. EARTH ENGINE FACTORS
# ------------------------------------------------------------

Map_factors = geemap.Map(
    center=[23.6850, 90.3563],
    zoom=7,
    height="750px"
)

Map_factors.add_basemap("SATELLITE")


ee_vis = {
    "min": 0,
    "max": 1,
    "palette": [
        "000004",
        "3b0f70",
        "8c2981",
        "de4968",
        "fe9f6d",
        "fcfdbf"
    ]
}


for name, image in factors.items():

    if isinstance(image, ee.image.Image):

        print(f"Adding Earth Engine factor: {name}")

        Map_factors.addLayer(
            image,
            ee_vis,
            factor_titles[name]
        )


# ------------------------------------------------------------
# Bangladesh boundary
# ------------------------------------------------------------

boundary = ee.Image().byte().paint(
    featureCollection=ee.FeatureCollection([
        ee.Feature(ee_aoi)
    ]),
    color=1,
    width=2
)

Map_factors.addLayer(
    boundary,
    {"palette": ["red"]},
    "Bangladesh Boundary"
)

Map_factors.addLayerControl()

Map_factors

In [ ]:
# ============================================================
# EE FACTOR -> LOCAL XARRAY
# ROBUST VERSION: NO CUSTOM CRS DURING EE DOWNLOAD
# ============================================================

from pathlib import Path
import rioxarray as rxr
import geemap
import os

EE_FACTOR_DIR = Path("outputs/ee_factors")
EE_FACTOR_DIR.mkdir(parents=True, exist_ok=True)


def ee_factor_to_xarray(ee_img, name, scale=1000):

    out_tif = EE_FACTOR_DIR / f"{name}.tif"

    print(f"\nProcessing: {name}")

    # --------------------------------------------------------
    # Remove failed / empty previous file
    # --------------------------------------------------------
    if out_tif.exists() and out_tif.stat().st_size == 0:
        out_tif.unlink()

    # --------------------------------------------------------
    # Inspect native EE projection
    # --------------------------------------------------------
    proj = ee_img.select(0).projection().getInfo()

    print("  EE native CRS:", proj.get("crs"))
    print("  EE transform :", proj.get("transform"))

    # --------------------------------------------------------
    # Export WITHOUT TARGET_EPSG
    # Let Earth Engine use a valid projection
    # --------------------------------------------------------
    if not out_tif.exists():

        print("  Downloading from Earth Engine...")

        geemap.ee_export_image(
            ee_img,
            filename=str(out_tif),
            region=ee_aoi,
            scale=scale,
            file_per_band=False,
        )

    else:
        print("  Existing file found:", out_tif)

    # --------------------------------------------------------
    # Verify
    # --------------------------------------------------------
    if not out_tif.exists():
        raise RuntimeError(
            f"Export failed. File was not created: {out_tif}"
        )

    if out_tif.stat().st_size == 0:
        raise RuntimeError(
            f"Export produced an empty file: {out_tif}"
        )

    # --------------------------------------------------------
    # Open with rioxarray
    # --------------------------------------------------------
    da = rxr.open_rasterio(
        out_tif,
        masked=True
    )

    if "band" in da.dims:
        da = da.squeeze("band", drop=True)

    da = da.astype("float32")

    print("  Local shape:", da.shape)
    print("  Local CRS  :", da.rio.crs)
    print("  READY")

    return da

In [ ]:
nightlight_local = ee_factor_to_xarray(
    nightlight_score,
    "nightlight_score",
    scale=500,
)

In [ ]:
# ============================================================
# POPULATION: EXPORT RAW GHSL 2025, THEN NORMALIZE LOCALLY
# ============================================================

from pathlib import Path
import numpy as np
import rioxarray as rxr
import geemap

EE_FACTOR_DIR = Path("outputs/ee_factors")
EE_FACTOR_DIR.mkdir(parents=True, exist_ok=True)

pop_tif = EE_FACTOR_DIR / "ghsl_population_2025.tif"


# ------------------------------------------------------------
# 1. Load RAW GHSL population
# ------------------------------------------------------------

population_raw = (
    ee.Image("JRC/GHSL/P2023A/GHS_POP/2025")
    .select("population_count")
    .clip(ee_aoi)
)

print(
    "Population bands:",
    population_raw.bandNames().getInfo()
)


# ------------------------------------------------------------
# 2. Export using explicit valid CRS
# ------------------------------------------------------------
# EPSG:4326 is safe for EE download.
# Reprojection to Sentinel grid happens later.

if not pop_tif.exists():

    print("Downloading GHSL population 2025...")

    geemap.ee_export_image(
        population_raw,
        filename=str(pop_tif),
        scale=1000,          # suitable for national urban-factor model
        crs="EPSG:4326",
        region=ee_aoi,
        file_per_band=False,
    )

else:
    print("Existing population raster found.")


# ------------------------------------------------------------
# 3. Verify file
# ------------------------------------------------------------

if not pop_tif.exists():
    raise RuntimeError(
        "Population export failed: file was not created."
    )


# ------------------------------------------------------------
# 4. Open locally
# ------------------------------------------------------------

population_local_raw = rxr.open_rasterio(
    pop_tif,
    masked=True
)

if "band" in population_local_raw.dims:
    population_local_raw = population_local_raw.squeeze(
        "band",
        drop=True
    )

population_local_raw = population_local_raw.astype("float32")

print("Population raster loaded.")
print("Shape:", population_local_raw.shape)
print("CRS:", population_local_raw.rio.crs)

In [ ]:
# ============================================================
# LOCAL POPULATION NORMALIZATION 0-1
# ============================================================

pop = population_local_raw.where(
    population_local_raw > 0
)

# log1p transformation
pop_log = np.log1p(pop)

# Compute percentile values
p2 = float(
    pop_log.quantile(0.02, skipna=True).compute()
)

p98 = float(
    pop_log.quantile(0.98, skipna=True).compute()
)

print("Population log P2 :", p2)
print("Population log P98:", p98)


if p98 <= p2:
    raise RuntimeError(
        "Invalid population percentile range."
    )


population_local = (
    (pop_log - p2) / (p98 - p2)
).clip(
    min=0,
    max=1
).astype(
    "float32"
).rename(
    "population_score"
)

print("population_local READY.")
print(
    "Range:",
    float(population_local.min(skipna=True).compute()),
    "to",
    float(population_local.max(skipna=True).compute()),
)

In [ ]:
topography_local = ee_factor_to_xarray(
    topography_score,
    "topography_score",
    scale=500,
)

In [ ]:
# ============================================================
# 32A. FINAL FACTOR DICTIONARY
# ============================================================

factors = {
    "builtup": builtup_probability,
    "nightlight": nightlight_local,
    "population": population_local,
    "road": road_score,
    "poi": poi_score,
    "topography": topography_local,
}

print("FINAL FACTOR TYPES")
print("-" * 70)

for name, da in factors.items():
    print(
        f"{name:12s}: "
        f"{type(da).__module__}.{type(da).__name__}"
    )

In [ ]:
# ============================================================
# 32B. CHECK / DEFINE WEIGHTS
# ============================================================

if "WEIGHTS" not in globals():

    WEIGHTS = {
        "builtup": 0.30,
        "nightlight": 0.20,
        "population": 0.20,
        "road": 0.10,
        "poi": 0.10,
        "topography": 0.10,
    }

print("WEIGHTS")
print("-" * 40)

for k, v in WEIGHTS.items():
    print(f"{k:12s}: {v:.2f}")

print("-" * 40)
print("TOTAL:", sum(WEIGHTS.values()))

if not np.isclose(sum(WEIGHTS.values()), 1.0):
    raise ValueError("Weights must sum to 1.0")

In [ ]:
# ============================================================
# 32C. ALIGN ALL SIX FACTORS TO SENTINEL TEMPLATE
# ============================================================

print("\nAligning factors to Sentinel template...")
print("-" * 60)

aligned_factors = {}

for name, da in factors.items():

    print(f"Processing {name}...")

    # Must be local raster / DataArray
    if not hasattr(da, "rio"):
        raise TypeError(
            f"{name} is not a rioxarray DataArray. "
            f"Current type = {type(da)}"
        )

    # Set spatial dimensions
    da = da.rio.set_spatial_dims(
        x_dim="x",
        y_dim="y",
        inplace=False
    )

    # --------------------------------------------------------
    # DO NOT overwrite an existing CRS
    # --------------------------------------------------------
    if da.rio.crs is None:

        if name in ["builtup", "road", "poi"]:
            da = da.rio.write_crs(
                f"EPSG:{TARGET_EPSG}",
                inplace=False
            )

        else:
            raise ValueError(
                f"{name} has no CRS. "
                "Do not assign TARGET_EPSG blindly."
            )

    # --------------------------------------------------------
    # Built-up is already template-based
    # --------------------------------------------------------
    if name == "builtup":

        aligned = da

    else:

        aligned = align_to_template(
            da,
            "bilinear"
        )

    # --------------------------------------------------------
    # Keep score within 0–1
    # --------------------------------------------------------
    aligned = (
        aligned
        .clip(min=0, max=1)
        .astype("float32")
    )

    aligned_factors[name] = aligned

    print(
        "   READY |",
        "shape =", aligned.shape,
        "| CRS =", aligned.rio.crs
    )

print("\nALL SIX FACTORS ALIGNED.")

In [ ]:
# ============================================================
# 32D. VERIFY EXACT GRID MATCH
# ============================================================

reference = aligned_factors["builtup"]

print("\nGRID CHECK")
print("-" * 60)

for name, da in aligned_factors.items():

    same_shape = da.shape == reference.shape

    same_x = np.array_equal(
        da.x.values,
        reference.x.values
    )

    same_y = np.array_equal(
        da.y.values,
        reference.y.values
    )

    print(
        f"{name:12s} | "
        f"shape={same_shape} | "
        f"x={same_x} | "
        f"y={same_y}"
    )

    if not (same_shape and same_x and same_y):
        raise RuntimeError(
            f"{name} does not exactly match the template grid."
        )

print("\nEXACT GRID MATCH CONFIRMED.")

In [ ]:
# ============================================================
# 32E. MULTI-FACTOR URBAN SCORE
# ============================================================

print("\nCalculating Multi-Factor Urban Score...")

urban_score = None

for name, weight in WEIGHTS.items():

    component = (
        aligned_factors[name] * float(weight)
    )

    if urban_score is None:
        urban_score = component
    else:
        urban_score = urban_score + component


urban_score = (
    urban_score
    .clip(min=0, max=1)
    .astype("float32")
    .rename("Urban_Score")
)

print("Weighted score calculated.")

In [ ]:
# ============================================================
# FIND ALL PROJ DATABASES
# ============================================================

import os
from pathlib import Path
import pyproj

env_root = Path(os.environ["CONDA_PREFIX"])

print("CONDA_PREFIX:")
print(env_root)

print("\nCurrent PROJ_DATA:")
print(os.environ.get("PROJ_DATA"))

print("\nCurrent PROJ_LIB:")
print(os.environ.get("PROJ_LIB"))

print("\nPyProj data directory:")
print(pyproj.datadir.get_data_dir())

print("\nSearching proj.db files...\n")

for p in env_root.rglob("proj.db"):
    print(p)

In [ ]:
# ============================================================
# FIX PROJ DATABASE PATH FOR RASTERIO
# RUN THIS BEFORE rasterio / geopandas / rioxarray IMPORTS
# ============================================================

import os
from pathlib import Path

CONDA_PREFIX = Path(os.environ["CONDA_PREFIX"])

RASTERIO_PROJ = (
    CONDA_PREFIX
    / "Lib"
    / "site-packages"
    / "rasterio"
    / "proj_data"
)

proj_db = RASTERIO_PROJ / "proj.db"

if not proj_db.exists():
    raise RuntimeError(
        f"Rasterio proj.db not found: {proj_db}"
    )

# Override the broken conda-level PROJ database
os.environ["PROJ_DATA"] = str(RASTERIO_PROJ)

# Remove legacy setting if present
os.environ.pop("PROJ_LIB", None)

print("PROJ_DATA fixed to:")
print(os.environ["PROJ_DATA"])

In [ ]:
# ============================================================
# SAVE EXPENSIVE CURRENT FACTORS
# WITHOUT RESTARTING THE KERNEL
# ============================================================

from pathlib import Path

CACHE_DIR = Path("cache_final")
CACHE_DIR.mkdir(exist_ok=True)

save_vars = {
    "builtup_probability": builtup_probability,
    "road_score": road_score,
    "poi_score": poi_score,
}

for name, da in save_vars.items():

    path = CACHE_DIR / f"{name}.nc"

    print(f"Saving {name}...")

    try:
        da.to_netcdf(path)
        print("  SAVED:", path)

    except Exception as e:
        print("  FAILED:", e)

print("\nCACHE STEP FINISHED.")

In [ ]:
from pathlib import Path

CACHE_DIR = Path("cache_final")

for fname in [
    "builtup_probability.nc",
    "road_score.nc",
    "poi_score.nc",
]:
    p = CACHE_DIR / fname

    print(
        fname,
        "EXISTS" if p.exists() else "MISSING",
        "|",
        f"{p.stat().st_size / 1024 / 1024:.2f} MB"
        if p.exists() else ""
    )

In [ ]:
# ============================================================
# SAVE BUILT-UP PROBABILITY SAFELY
# ============================================================

from pathlib import Path

CACHE_DIR = Path("cache_final")
CACHE_DIR.mkdir(parents=True, exist_ok=True)

out_file = CACHE_DIR / "builtup_probability.nc"

print("Saving builtup_probability...")
print("Shape:", builtup_probability.shape)

try:
    builtup_probability.to_netcdf(
        out_file,
        engine="scipy"
    )

    print("\nSAVE COMPLETE")
    print("File:", out_file)
    print(
        "Size:",
        round(out_file.stat().st_size / 1024**2, 2),
        "MB"
    )

except Exception as e:
    print("\nSAVE FAILED")
    print(type(e).__name__, ":", e)

In [ ]:
# 34. URBAN CENTER MASK
urban_mask = (
    urban_score_bd >= URBAN_SCORE_THRESHOLD
).astype("uint8").rename("Urban_Center_Mask")

mask_preview = (
    urban_mask
    .coarsen(x=4, y=4, boundary="trim")
    .max()
    .compute()
)

plt.figure(figsize=(8, 9))
mask_preview.plot()
plt.title(
    f"Urban Center Candidate Mask "
    f"(Score ≥ {URBAN_SCORE_THRESHOLD})"
)
plt.axis("equal")
plt.show()


In [ ]:
# 35. CONNECTED COMPONENTS + MINIMUM PATCH SIZE

mask_np = urban_mask.compute().values.astype(bool)

# 8-neighbour connectivity
structure = np.ones((3, 3), dtype="uint8")

labels, n_components = ndimage.label(
    mask_np,
    structure=structure,
)

pixel_area_km2 = (
    RESOLUTION_M * RESOLUTION_M
) / 1_000_000.0

min_pixels = max(
    1,
    int(np.ceil(
        MIN_PATCH_AREA_KM2 / pixel_area_km2
    )),
)

counts = np.bincount(labels.ravel())

keep_labels = np.where(
    counts >= min_pixels
)[0]

keep_labels = keep_labels[
    keep_labels != 0
]

filtered_np = np.isin(
    labels,
    keep_labels,
)

filtered_mask = xr.DataArray(
    filtered_np.astype("uint8"),
    dims=("y", "x"),
    coords={
        "y": urban_mask.y.values,
        "x": urban_mask.x.values,
    },
    name="Urban_Center_Filtered",
)

filtered_mask = (
    filtered_mask.rio
    .set_spatial_dims(
        x_dim="x",
        y_dim="y",
        inplace=False,
    )
    .rio.write_crs(
        f"EPSG:{TARGET_EPSG}",
        inplace=False,
    )
)

print("Initial components:", n_components)
print("Minimum patch:", MIN_PATCH_AREA_KM2, "km²")
print("Minimum pixels:", min_pixels)
print("Retained components:", len(keep_labels))


In [ ]:
# 36. POLYGONIZE URBAN CENTER CANDIDATES

records = []

for geom, value in shapes(
    filtered_mask.values,
    mask=filtered_mask.values.astype(bool),
    transform=filtered_mask.rio.transform(),
):
    if int(value) == 1:
        records.append({
            "geometry": shape(geom),
            "class": 1,
        })

urban_polygons = gpd.GeoDataFrame(
    records,
    crs=f"EPSG:{TARGET_EPSG}",
)

if urban_polygons.empty:
    print("No candidate polygons found.")
else:
    urban_polygons["area_km2"] = (
        urban_polygons.geometry.area
        / 1_000_000.0
    )

    urban_polygons = (
        urban_polygons
        .reset_index(drop=True)
    )

    urban_polygons["urban_id"] = np.arange(
        1,
        len(urban_polygons) + 1,
    )

    print(
        "Candidate urban-center polygons:",
        len(urban_polygons),
    )


In [ ]:
# 37. ZONAL MEAN URBAN SCORE + POPULATION DENSITY

def zonal_mean(gdf, da, field_name):
    arr = da.compute().values
    transform = da.rio.transform()

    result = []

    for geom in gdf.geometry:
        zone = rasterize(
            [(geom, 1)],
            out_shape=arr.shape,
            transform=transform,
            fill=0,
            dtype="uint8",
            all_touched=False,
        ).astype(bool)

        vals = arr[zone]
        vals = vals[np.isfinite(vals)]

        result.append(
            float(vals.mean())
            if vals.size
            else np.nan
        )

    gdf[field_name] = result
    return gdf


if not urban_polygons.empty:
    urban_polygons = zonal_mean(
        urban_polygons,
        urban_score_bd,
        "mean_score",
    )

    population_density_bd = clip_bd(
        align_to_template(
            population_density,
            "bilinear",
        )
    )

    urban_polygons = zonal_mean(
        urban_polygons,
        population_density_bd,
        "mean_popden",
    )

    display(
        urban_polygons[
            [
                "urban_id",
                "area_km2",
                "mean_score",
                "mean_popden",
            ]
        ].head(20)
    )


In [ ]:
# 38. FINAL URBAN-CENTER FILTERING
if urban_polygons.empty:
    final_urban_centers = urban_polygons.copy()
else:
    final_urban_centers = urban_polygons[
        (urban_polygons["area_km2"] >= MIN_PATCH_AREA_KM2)
        & (
            urban_polygons["mean_score"]
            >= MIN_MEAN_URBAN_SCORE
        )
        & (
            urban_polygons["mean_popden"]
            >= MIN_MEAN_POP_DENSITY
        )
    ].copy()

    final_urban_centers = (
        final_urban_centers
        .reset_index(drop=True)
    )

    final_urban_centers["urban_id"] = np.arange(
        1,
        len(final_urban_centers) + 1,
    )

print(
    "Final urban centers:",
    len(final_urban_centers),
)


In [ ]:
# 39. FINAL INTERACTIVE MAP IN VS CODE

final_map = leafmap.Map(
    center=[23.6850, 90.3563],
    zoom=7,
    height="750px",
)

final_map.add_gdf(
    aoi,
    layer_name="Bangladesh",
    style={
        "color": "black",
        "weight": 2,
        "fillOpacity": 0,
    },
)

if not final_urban_centers.empty:
    final_map.add_gdf(
        final_urban_centers.to_crs(4326),
        layer_name="Final Urban Centers",
        style={
            "color": "red",
            "weight": 1,
            "fillColor": "red",
            "fillOpacity": 0.50,
        },
    )

final_map


In [ ]:
# 40. SAVE ONLY FINAL / IMPORTANT OUTPUTS
#
# Source data were queried online.
# These are your analysis outputs.

outputs = {
    "Builtup_Probability_2025_100m.tif":
        clip_bd(builtup_probability),

    "NightLight_Score_2025_100m.tif":
        clip_bd(aligned_factors["nightlight"]),

    "Population_Score_2025_100m.tif":
        clip_bd(aligned_factors["population"]),

    "Road_Score_2025_100m.tif":
        clip_bd(aligned_factors["road"]),

    "POI_Service_Score_2025_100m.tif":
        clip_bd(aligned_factors["poi"]),

    "Topography_Score_100m.tif":
        clip_bd(aligned_factors["topography"]),

    "Urban_Score_2025_100m.tif":
        urban_score_bd,

    "Urban_Center_Mask_2025_100m.tif":
        filtered_mask,
}

for filename, da in outputs.items():
    path = OUTPUT_DIR / filename
    print("Writing:", path)

    da.rio.to_raster(
        path,
        compress="DEFLATE",
        tiled=True,
        BIGTIFF="IF_SAFER",
    )

gpkg_path = (
    OUTPUT_DIR
    / "Bangladesh_Urban_Centers_2025.gpkg"
)

final_urban_centers.to_file(
    gpkg_path,
    layer="urban_centers",
    driver="GPKG",
)

print("\nDONE")
print("Final vector:", gpkg_path)
print("Output folder:", OUTPUT_DIR)


## Final logic

`Sentinel built-up probability × 0.30`  
`+ VIIRS persistent-light score × 0.20`  
`+ GHSL population score × 0.20`  
`+ Overture road score × 0.10`  
`+ Overture POI/service score × 0.10`  
`+ topography score × 0.10`  
`= Urban Score (0–1)`

Then:

**Urban Score threshold → connected components → minimum area → mean score + mean population density filter → final urban-center polygons**
